# 🎭 image2live2d — Google Colab

Chuyển đổi character art thành **2D puppet có thể rig** (blink · lip-sync · head-turn · hair physics).

**Outputs:** `.inp` (nijilive) · `.moc3` (Live2D) · `.cmo3` (Cubism Editor)

---
### Trước khi chạy:
1. **Runtime → Change runtime type → T4 GPU** (hoặc A100 nếu có Colab Pro)
2. Chạy từng cell theo thứ tự
3. Cell 1-3 chỉ cần chạy **một lần** per session (cài đặt ~10-15 phút)
4. Cell 4+ dùng để convert ảnh

---
> **Flat image PNG/JPG** cần See-through service (Cell 2-3).  
> **PSD / ZIP layers** không cần GPU service, convert ngay sau Cell 1.

In [1]:
# ============================================================
# CELL 1 — Cài đặt image2live2d
# Thời gian: ~2-3 phút
# ============================================================
import subprocess, sys

# Clone repo
!git clone https://github.com/giantiensinh/imagelive2d.git /content/image2live2d 2>/dev/null || \
  (cd /content/image2live2d && git pull)

# Cài package + tất cả dependencies
%cd /content/image2live2d
!pip install -q -e '.[decompose,preprocess,mesh,rig,app]'

# Kiểm tra
!python -c "import image2live2d; print('✅ image2live2d', image2live2d.__version__ if hasattr(image2live2d, '__version__') else 'OK')"
!python -m image2live2d --sample --fullbody -o /tmp/test.inp && echo '✅ Pipeline test: PASS'

/content/image2live2d
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.7/835.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 111.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 MB 11.2 MB/s eta 0:00:00
  Building editable 

In [2]:
# ============================================================
# CELL 2 & 3 (Gộp) — Cài đặt, Khởi động See-through & Tải Model Weights
# Thời gian: ~10-15 phút (Tải weights ~12GB)
# ============================================================
import os, subprocess, time, requests

# 1. KIỂM TRA GPU
print("=== 1. Kiểm tra phần cứng GPU ===")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. CLONE SEE-THROUGH & SETUP
print("\n=== 2. Cài đặt See-through source code & OpenCV dependencies ===")
SEE_THROUGH_COMMIT = "58a1cb11d13f85acec9bbddb8cd4b6487843d4cf"
!git clone https://github.com/shitagaki-lab/see-through.git /opt/see-through 2>/dev/null || true
!cd /opt/see-through && git fetch --quiet origin {SEE_THROUGH_COMMIT} 2>/dev/null || true
!cd /opt/see-through && git checkout --quiet {SEE_THROUGH_COMMIT} 2>/dev/null || echo 'Using default HEAD'
!cd /opt/see-through && ln -sf common/assets assets 2>/dev/null || true
!apt-get install -qq -y libgl1 libglib2.0-0 > /dev/null

# 3. CÀI ĐẶT PYTORCH CUDA & DEPENDENCIES
print("\n=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===")
!pip install -q torch==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r /opt/see-through/requirements.txt
!pip install -q fastapi 'uvicorn[standard]' huggingface_hub

# 4. THIẾT LẬP APP SERVICE
print("\n=== 4. Cấu hình và Vá lỗi tương thích (Patches) ===")
!cp /content/image2live2d/service/seethrough/app.py /opt/
!cp /content/image2live2d/service/seethrough/pipeline.py /opt/
!cp /content/image2live2d/service/seethrough/model_manager.py /opt/
!mkdir -p /opt/patches
!cp /content/image2live2d/service/seethrough/patches/*.py /opt/patches/
!python /opt/patches/guard_empty_head_crop.py /opt/see-through/common/utils/inference_utils.py
!python /opt/patches/guard_missing_part_pngs.py /opt/see-through/common/utils/io_utils.py

# 5. KHỞI ĐỘNG BACKGROUND SERVICE
print("\n=== 5. Đang khởi động background service... ===")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("/opt/see-through/workspace", exist_ok=True)

env = os.environ.copy()
env.update({
    "SEE_THROUGH_DIR": "/opt/see-through",
    "MODEL_DIR": MODEL_DIR,
    "HF_HOME": MODEL_DIR + "/cache",
    "MANAGED_MODELS": "1",
    "SEE_THROUGH_TIMEOUT": "1800",
})

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8765"],
    cwd="/opt", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Chờ service phản hồi
for _ in range(30):
    try:
        r = requests.get("http://localhost:8765/health", timeout=2)
        if r.status_code == 200:
            print(" ✅ Service đã hoạt động thành công!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ Lỗi: Service không thể khởi chạy. Vui lòng chạy lại cell này.")

# 6. TẢI MODEL WEIGHTS (~12GB)
print("\n=== 6. Kiểm tra và Tải Model Weights (~12GB) ===")
status = requests.get("http://localhost:8765/setup").json()
print(f"GPU hiện tại: {status.get('gpu_name', 'N/A')} — VRAM khả dụng: {status.get('vram_gb', '?')} GB")

if not status.get("models_installed"):
    print("📥 Đang tải weights từ Hugging Face... Vui lòng không tắt Colab.")
    requests.post("http://localhost:8765/setup/models")
    while True:
        s = requests.get("http://localhost:8765/setup").json()
        state = s.get("state", "?")
        msg = s.get("message", "")
        print(f"  [{state.upper()}] {msg[:80]}", flush=True)
        if state in ("ready", "error"):
            break
        time.sleep(15)
else:
    print("✅ Bộ weights đã tồn tại sẵn từ trước!")

print("\n🎉 TẤT CẢ ĐÃ SẴN SÀNG! Bạn có thể chuyển ngay xuống CELL 4A bên dưới để tải ảnh lên và convert.")

=== 1. Kiểm tra phần cứng GPU ===
Tesla T4, 15360 MiB

=== 2. Cài đặt See-through source code & OpenCV dependencies ===

=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 48.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 41.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 92.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 57.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 103.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 138.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 79.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 117.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 57.7 MB/s eta 0:00:00
  

In [3]:
# ============================================================
# CELL 2 & 3 (Gộp) — Cài đặt, Khởi động See-through & Tải Model Weights
# Thời gian: ~10-15 phút (Tải weights ~12GB)
# ============================================================
import os, subprocess, time, requests

# 1. KIỂM TRA GPU
print("=== 1. Kiểm tra phần cứng GPU ===")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. CLONE SEE-THROUGH & SETUP
print("\n=== 2. Cài đặt See-through source code & OpenCV dependencies ===")
SEE_THROUGH_COMMIT = "58a1cb11d13f85acec9bbddb8cd4b6487843d4cf"
!git clone https://github.com/shitagaki-lab/see-through.git /opt/see-through 2>/dev/null || true
!cd /opt/see-through && git fetch --quiet origin {SEE_THROUGH_COMMIT} 2>/dev/null || true
!cd /opt/see-through && git checkout --quiet {SEE_THROUGH_COMMIT} 2>/dev/null || echo 'Using default HEAD'
!cd /opt/see-through && ln -sf common/assets assets 2>/dev/null || true
!apt-get install -qq -y libgl1 libglib2.0-0 > /dev/null

# 3. CÀI ĐẶT PYTORCH CUDA & DEPENDENCIES
print("\n=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===")
!pip install -q torch==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r /opt/see-through/requirements.txt
!pip install -q fastapi 'uvicorn[standard]' huggingface_hub

# 4. THIẾT LẬP APP SERVICE
print("\n=== 4. Cấu hình và Vá lỗi tương thích (Patches) ===")
!cp /content/image2live2d/service/seethrough/app.py /opt/
!cp /content/image2live2d/service/seethrough/pipeline.py /opt/
!cp /content/image2live2d/service/seethrough/model_manager.py /opt/
!mkdir -p /opt/patches
!cp /content/image2live2d/service/seethrough/patches/*.py /opt/patches/
!python /opt/patches/guard_empty_head_crop.py /opt/see-through/common/utils/inference_utils.py
!python /opt/patches/guard_missing_part_pngs.py /opt/see-through/common/utils/io_utils.py

# 5. KHỞI ĐỘNG BACKGROUND SERVICE
print("\n=== 5. Đang khởi động background service... ===")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("/opt/see-through/workspace", exist_ok=True)

env = os.environ.copy()
env.update({
    "SEE_THROUGH_DIR": "/opt/see-through",
    "MODEL_DIR": MODEL_DIR,
    "HF_HOME": MODEL_DIR + "/cache",
    "MANAGED_MODELS": "1",
    "SEE_THROUGH_TIMEOUT": "1800",
})

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8765"],
    cwd="/opt", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Chờ service phản hồi
for _ in range(30):
    try:
        r = requests.get("http://localhost:8765/health", timeout=2)
        if r.status_code == 200:
            print(" ✅ Service đã hoạt động thành công!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ Lỗi: Service không thể khởi chạy. Vui lòng chạy lại cell này.")

# 6. TẢI MODEL WEIGHTS (~12GB)
print("\n=== 6. Kiểm tra và Tải Model Weights (~12GB) ===")
status = requests.get("http://localhost:8765/setup").json()
print(f"GPU hiện tại: {status.get('gpu_name', 'N/A')} — VRAM khả dụng: {status.get('vram_gb', '?')} GB")

if not status.get("models_installed"):
    print("📥 Đang tải weights từ Hugging Face... Vui lòng không tắt Colab.")
    requests.post("http://localhost:8765/setup/models")
    while True:
        s = requests.get("http://localhost:8765/setup").json()
        state = s.get("state", "?")
        msg = s.get("message", "")
        print(f"  [{state.upper()}] {msg[:80]}", flush=True)
        if state in ("ready", "error"):
            break
        time.sleep(15)
else:
    print("✅ Bộ weights đã tồn tại sẵn từ trước!")

print("\n🎉 TẤT CẢ ĐÃ SẴN SÀNG! Bạn có thể chuyển ngay xuống CELL 4A bên dưới để tải ảnh lên và convert.")

=== 1. Kiểm tra phần cứng GPU ===
Tesla T4, 15360 MiB

=== 2. Cài đặt See-through source code & OpenCV dependencies ===

=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===
ERROR: ./common is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).

=== 4. Cấu hình và Vá lỗi tương thích (Patches) ===
guard_empty_head_crop: /opt/see-through/common/utils/inference_utils.py already patched (no-op)
guard_missing_part_pngs: /opt/see-through/common/utils/io_utils.py already patched (no-op)

=== 5. Đang khởi động background service... ===
 ✅ Service đã hoạt động thành công!

=== 6. Kiểm tra và Tải Model Weights (~12GB) ===
GPU hiện tại: Tesla T4 — VRAM khả dụng: 14.6 GB
✅ Bộ weights đã tồn tại sẵn từ trước!

🎉 TẤ

In [4]:
# ============================================================
# CELL 2 & 3 (Gộp) — Cài đặt, Khởi động See-through & Tải Model Weights
# Thời gian: ~10-15 phút (Tải weights ~12GB)
# ============================================================
import os, subprocess, time, requests

# 1. KIỂM TRA GPU
print("=== 1. Kiểm tra phần cứng GPU ===")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# 2. CLONE SEE-THROUGH & SETUP
print("\n=== 2. Cài đặt See-through source code & OpenCV dependencies ===")
SEE_THROUGH_COMMIT = "58a1cb11d13f85acec9bbddb8cd4b6487843d4cf"
!git clone https://github.com/shitagaki-lab/see-through.git /opt/see-through 2>/dev/null || true
!cd /opt/see-through && git fetch --quiet origin {SEE_THROUGH_COMMIT} 2>/dev/null || true
!cd /opt/see-through && git checkout --quiet {SEE_THROUGH_COMMIT} 2>/dev/null || echo 'Using default HEAD'
!cd /opt/see-through && ln -sf common/assets assets 2>/dev/null || true
!apt-get install -qq -y libgl1 libglib2.0-0 > /dev/null

# 3. CÀI ĐẶT PYTORCH CUDA & DEPENDENCIES
print("\n=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===")
!pip install -q torch==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r /opt/see-through/requirements.txt
!pip install -q fastapi 'uvicorn[standard]' huggingface_hub

# 4. THIẾT LẬP APP SERVICE
print("\n=== 4. Cấu hình và Vá lỗi tương thích (Patches) ===")
!cp /content/image2live2d/service/seethrough/app.py /opt/
!cp /content/image2live2d/service/seethrough/pipeline.py /opt/
!cp /content/image2live2d/service/seethrough/model_manager.py /opt/
!mkdir -p /opt/patches
!cp /content/image2live2d/service/seethrough/patches/*.py /opt/patches/
!python /opt/patches/guard_empty_head_crop.py /opt/see-through/common/utils/inference_utils.py
!python /opt/patches/guard_missing_part_pngs.py /opt/see-through/common/utils/io_utils.py

# 5. KHỞI ĐỘNG BACKGROUND SERVICE
print("\n=== 5. Đang khởi động background service... ===")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("/opt/see-through/workspace", exist_ok=True)

env = os.environ.copy()
env.update({
    "SEE_THROUGH_DIR": "/opt/see-through",
    "MODEL_DIR": MODEL_DIR,
    "HF_HOME": MODEL_DIR + "/cache",
    "MANAGED_MODELS": "1",
    "SEE_THROUGH_TIMEOUT": "1800",
})

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8765"],
    cwd="/opt", env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Chờ service phản hồi
for _ in range(30):
    try:
        r = requests.get("http://localhost:8765/health", timeout=2)
        if r.status_code == 200:
            print(" ✅ Service đã hoạt động thành công!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ Lỗi: Service không thể khởi chạy. Vui lòng chạy lại cell này.")

# 6. TẢI MODEL WEIGHTS (~12GB)
print("\n=== 6. Kiểm tra và Tải Model Weights (~12GB) ===")
status = requests.get("http://localhost:8765/setup").json()
print(f"GPU hiện tại: {status.get('gpu_name', 'N/A')} — VRAM khả dụng: {status.get('vram_gb', '?')} GB")

if not status.get("models_installed"):
    print("📥 Đang tải weights từ Hugging Face... Vui lòng không tắt Colab.")
    requests.post("http://localhost:8765/setup/models")
    while True:
        s = requests.get("http://localhost:8765/setup").json()
        state = s.get("state", "?")
        msg = s.get("message", "")
        print(f"  [{state.upper()}] {msg[:80]}", flush=True)
        if state in ("ready", "error"):
            break
        time.sleep(15)
else:
    print("✅ Bộ weights đã tồn tại sẵn từ trước!")

print("\n🎉 TẤT CẢ ĐÃ SẴN SÀNG! Bạn có thể chuyển ngay xuống CELL 4A bên dưới để tải ảnh lên và convert.")

=== 1. Kiểm tra phần cứng GPU ===
Tesla T4, 15360 MiB

=== 2. Cài đặt See-through source code & OpenCV dependencies ===

=== 3. Cài đặt PyTorch CUDA & Python packages (Sẽ mất ít phút) ===
ERROR: ./common is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).

=== 4. Cấu hình và Vá lỗi tương thích (Patches) ===
guard_empty_head_crop: /opt/see-through/common/utils/inference_utils.py already patched (no-op)
guard_missing_part_pngs: /opt/see-through/common/utils/io_utils.py already patched (no-op)

=== 5. Đang khởi động background service... ===
 ✅ Service đã hoạt động thành công!

=== 6. Kiểm tra và Tải Model Weights (~12GB) ===
GPU hiện tại: Tesla T4 — VRAM khả dụng: 14.6 GB
✅ Bộ weights đã tồn tại sẵn từ trước!

🎉 TẤ

In [12]:
# ============================================================
# CELL 3 — Start See-through service + Download model weights
# Bỏ qua nếu chỉ dùng PSD/ZIP
# Download lần đầu: ~12GB, mất ~5-10 phút
# ============================================================
import subprocess, time, requests, os

MODEL_DIR = "/content/models"  # lưu ở /content để persist trong session
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("/opt/see-through/workspace", exist_ok=True)

# Start service
env = os.environ.copy()
env.update({
    "SEE_THROUGH_DIR": "/opt/see-through",
    "MODEL_DIR": MODEL_DIR,
    "HF_HOME": MODEL_DIR + "/cache",
    "MANAGED_MODELS": "1",
    "SEE_THROUGH_TIMEOUT": "1800",
    # T4/A100: không cần giảm resolution hay inference steps
    # "RESOLUTION": "1024",  # default của See-through là 1280
})

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8765"],
    cwd="/opt", env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# Đợi service khởi động
print("Đang khởi động service...", end="")
for _ in range(30):
    try:
        r = requests.get("http://localhost:8765/health", timeout=2)
        if r.status_code == 200:
            print(" ✅ Service started!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    print("\n❌ Service không khởi động được. Kiểm tra log.")

# Download model weights (~12GB)
status = requests.get("http://localhost:8765/setup").json()
print(f"GPU: {status.get('gpu_name', 'N/A')} — VRAM: {status.get('vram_gb', '?')} GB")

if not status.get("models_installed"):
    print("Downloading See-through model weights (~12GB)...")
    requests.post("http://localhost:8765/setup/models")
    while True:
        s = requests.get("http://localhost:8765/setup").json()
        state = s.get("state", "?")
        msg = s.get("message", "")
        print(f"  [{state}] {msg[:80]}", flush=True)
        if state in ("ready", "error"):
            break
        time.sleep(15)
else:
    print("✅ Models đã có sẵn!")

print("\n✅ Seethrough service sẵn sàng tại http://localhost:8765")
print("   Set biến môi trường: IMAGE2LIVE2D_DECOMPOSE_URL=http://localhost:8765")

Đang khởi động service.... ✅ Service started!
GPU: Tesla T4 — VRAM: 14.6 GB
✅ Models đã có sẵn!

✅ Seethrough service sẵn sàng tại http://localhost:8765
   Set biến môi trường: IMAGE2LIVE2D_DECOMPOSE_URL=http://localhost:8765


---
## 🎨 Convert ảnh

Chọn một trong các cách dưới đây:

In [10]:
# ============================================================
# CELL 4A — Upload file và convert (PSD / ZIP / PNG / JPG)
# ============================================================
from google.colab import files
import os, shutil

print("Chọn file để upload từ máy tính của bạn (PSD, ZIP, PNG, JPG):")
uploaded = files.upload()

for filename, data in uploaded.items():
    upload_path = f"/content/{filename}"
    with open(upload_path, 'wb') as f:
        f.write(data)
    print(f"✅ Đã tải lên thành công: {filename} ({len(data)/1024/1024:.1f} MB)")

INPUT_FILE = list(uploaded.keys())[0] if uploaded else None
INPUT_PATH = f"/content/{INPUT_FILE}" if INPUT_FILE else None
OUTPUT_DIR = f"/content/output_{INPUT_FILE.rsplit('.', 1)[0]}" if INPUT_FILE else "/content/output"

print(f"\nĐường dẫn File nguồn: {INPUT_PATH}")
print(f"Thư mục kết xuất: {OUTPUT_DIR}")

Chọn file để upload từ máy tính của bạn (PSD, ZIP, PNG, JPG):


Saving 1788663167379.png to 1788663167379 (3).png
✅ Đã tải lên thành công: 1788663167379 (3).png (1.7 MB)

Đường dẫn File nguồn: /content/1788663167379 (3).png
Thư mục kết xuất: /content/output_1788663167379 (3)


In [11]:
# ============================================================
# CELL 4B — Chạy pipeline convert
# ============================================================
import subprocess, os

if INPUT_PATH is None:
    raise RuntimeError("Chưa upload file. Chạy Cell 4A trước.")

ext = INPUT_FILE.rsplit('.', 1)[-1].lower()
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set URL decompose service nếu dùng flat image
env = os.environ.copy()

# Build command
out_inp = f"{OUTPUT_DIR}/{INPUT_FILE.rsplit('.', 1)[0]}.inp"
out_live2d = f"{OUTPUT_DIR}/live2d"

if ext == 'psd':
    cmd = ["python", "-m", "image2live2d", "--psd", INPUT_PATH,
           "-o", out_inp, "--live2d", out_live2d]
elif ext == 'zip':
    cmd = ["python", "-m", "image2live2d", INPUT_PATH,
           "-o", out_inp, "--live2d", out_live2d]
else:  # flat image
    print("🖼️  Flat image — dùng See-through decompose service")
    env['IMAGE2LIVE2D_DECOMPOSE_URL'] = 'http://localhost:8765'
    cmd = ["python", "-m", "image2live2d", "--image", INPUT_PATH,
           "--decompose-url", "http://localhost:8765",
           "-o", out_inp, "--live2d", out_live2d]

print(f"\nChạy: {' '.join(cmd)}\n")
result = subprocess.run(cmd, cwd="/content/image2live2d", env=env,
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
else:
    print("\n✅ Convert thành công!")

🖼️  Flat image — dùng See-through decompose service

Chạy: python -m image2live2d --image /content/1788663167379 (3).png --decompose-url http://localhost:8765 -o /content/output_1788663167379 (3)/1788663167379 (3).inp --live2d /content/output_1788663167379 (3)/live2d

decomposing /content/1788663167379 (3).png via http://localhost:8765 (remote GPU)…

STDERR: error: decompose service unreachable at http://localhost:8765: [Errno 111] Connection refused



In [7]:
# ============================================================
# CELL 5 — Download kết quả
# ============================================================
from google.colab import files
import os, zipfile

output_zip = f"/content/output_{INPUT_FILE.rsplit('.', 1)[0]}.zip"

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_DIR):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, OUTPUT_DIR)
            zf.write(fpath, arcname)

size = os.path.getsize(output_zip) / 1024 / 1024
print(f"📦 Đóng gói: {output_zip} ({size:.1f} MB)")
print("Nội dung:")
with zipfile.ZipFile(output_zip) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:50s} {info.file_size/1024:.1f} KB")

files.download(output_zip)
print("\n✅ Download started!")

📦 Đóng gói: /content/output_1788663167379 (2).zip (0.0 MB)
Nội dung:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started!


In [8]:
# ============================================================
# CELL 6 (tùy chọn) — Batch convert nhiều file
# ============================================================
from google.colab import files
import os

# Upload nhiều file cùng lúc
print("Upload nhiều PSD/ZIP (chọn tất cả cùng lúc):")
uploaded_batch = files.upload()

BATCH_DIR = "/content/batch_input"
BATCH_OUT = "/content/batch_output"
os.makedirs(BATCH_DIR, exist_ok=True)

for filename, data in uploaded_batch.items():
    with open(f"{BATCH_DIR}/{filename}", 'wb') as f:
        f.write(data)

!python -m image2live2d --batch {BATCH_DIR} -o {BATCH_OUT}

# Download kết quả
import zipfile
batch_zip = "/content/batch_output.zip"
with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(BATCH_OUT):
        for fname in filenames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, os.path.relpath(fpath, BATCH_OUT))

files.download(batch_zip)
print("✅ Batch output downloaded!")

Upload nhiều PSD/ZIP (chọn tất cả cùng lúc):


Saving output_1788663167379 (2).zip to output_1788663167379 (2) (1).zip
error: no PSDs or layer dirs found under /content/batch_input


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Batch output downloaded!


In [9]:
# ============================================================
# CELL 7 (tùy chọn) — Web app với ngrok
# Mở web UI drag-drop trong browser
# ============================================================
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, os, time

# Nếu đã chạy See-through service ở Cell 3:
env = os.environ.copy()
env['IMAGE2LIVE2D_DECOMPOSE_URL'] = 'http://localhost:8765'

# Start web app trên port 8766
web_proc = subprocess.Popen(
    ["python", "-m", "image2live2d", "--serve", "--port", "8766"],
    cwd="/content/image2live2d", env=env,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# Expose ra ngoài
tunnel = ngrok.connect(8766)
print(f"\n🌐 Web app URL: {tunnel.public_url}")
print("Mở URL trên để dùng web interface drag-drop")
print("(Ctrl+C để dừng)")

ERROR:pyngrok.process.ngrok:t=2026-10-14T09:00:04+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-14T09:00:04+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
# ============================================================
# CELL 8 (debug) — Kiểm tra trạng thái
# ============================================================
import subprocess

print("=== GPU ===")
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv,noheader

print("\n=== Python packages ===")
!pip show image2live2d pydantic pillow psd-tools torch | grep -E 'Name|Version'

print("\n=== See-through service ===")
try:
    import requests
    r = requests.get('http://localhost:8765/setup', timeout=3)
    s = r.json()
    print(f"Status: {'✅ Ready' if s.get('ready') else '⚠️  ' + s.get('state','?')}")
    print(f"GPU: {s.get('gpu_name','N/A')} — VRAM: {s.get('vram_gb','?')}GB")
    print(f"Models: {'✅ Installed' if s.get('models_installed') else '❌ Not installed'}")
    print(f"Message: {s.get('message','')}")
except Exception as e:
    print(f"Service not running: {e}")

In [38]:
import os, sys, subprocess, time, requests, zipfile
from google.colab import files

# 1. Định nghĩa các đường dẫn
INPUT_FILE = '1788663167379 (3).png'
INPUT_PATH = f"/content/{INPUT_FILE}"
OUTPUT_DIR = f"/content/output_1788663167379"

# Khôi phục file gốc sạch trước để chuẩn bị môi trường chạy chuẩn
try:
    subprocess.run(["cp", "/content/image2live2d/service/seethrough/app.py", "/opt/app.py"], check=True)
    subprocess.run(["cp", "/content/image2live2d/service/seethrough/model_manager.py", "/opt/model_manager.py"], check=True)
    print("🔄 Đã khôi phục trạng thái mã nguồn gốc sạch!")
except Exception as e:
    print(f"⚠️ Không thể khôi phục file gốc: {e}")

# Sửa lỗi AttributeError triệt để bằng cách ép buộc reinstall torch với CUDA tương thích nếu phát hiện lỗi cuda attribute
try:
    import torch
    if not hasattr(torch, 'cuda') or not torch.cuda.is_available():
        print("⚠️ Phát hiện phiên bản torch bị lỗi (không hỗ trợ CUDA). Tiến hành cài đặt lại bản chuẩn...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.5.1+cu124", "torchvision==0.20.1+cu124", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
        print("🔥 Đã cài đặt lại Torch CUDA chuẩn thành công!")
    else:
        print("✅ Kiểm tra môi trường: Torch CUDA hoạt động bình thường trên Kernel chính!")
except Exception as e:
    print(f"⚠️ Lỗi kiểm tra/cài đặt Torch: {e}")

# Tạo môi trường cực kỳ sạch cho tiến trình con uvicorn
site_packages_paths = [p for p in sys.path if p and ('site-packages' in p or 'dist-packages' in p)]
colab_site_packages = "/usr/local/lib/python3.13/dist-packages"
for p in site_packages_paths:
    if os.path.exists(os.path.join(p, "torch")):
        colab_site_packages = p
        break

env_srv = os.environ.copy()
env_srv["PYTHONNOUSERSITE"] = "1"
env_srv["PYTHONPATH"] = f"/opt/see-through:/opt/see-through/common:{colab_site_packages}"
env_srv["PATH"] = os.path.dirname(sys.executable) + ":" + env_srv.get("PATH", "")
env_srv["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:/usr/local/nvidia/lib:/usr/local/nvidia/lib64:" + env_srv.get("LD_LIBRARY_PATH", "")
env_srv["NVIDIA_VISIBLE_DEVICES"] = "all"

# Tắt service cũ đang chạy trên port 8765 nếu có
try:
    subprocess.run(["fuser", "-k", "8765/tcp"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1.5)
except Exception:
    pass

# 3. Khởi động Service See-Through
print("🔄 Khởi động dịch vụ nền See-through...")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("/opt/see-through/workspace", exist_ok=True)

env_srv.update({
    "SEE_THROUGH_DIR": "/opt/see-through",
    "MODEL_DIR": MODEL_DIR,
    "HF_HOME": MODEL_DIR + "/cache",
    "MANAGED_MODELS": "1",
    "SEE_THROUGH_TIMEOUT": "1800",
})

# Ghi log ra file tạm để hiển thị khi có lỗi
log_file = open("/content/seethrough_service.log", "w")

# Khởi chạy uvicorn app bằng chính python interpreter hiện tại để đồng bộ hóa tuyệt đối
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8765"],
    cwd="/opt", env=env_srv, stdout=log_file, stderr=log_file
)

for i in range(30):
    try:
        r = requests.get("http://localhost:8765/health", timeout=2)
        if r.status_code == 200:
            print("✅ Service đã khởi động thành công!")
            break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)
else:
    log_file.close()
    with open("/content/seethrough_service.log", "r") as f:
        print("\n❌ LỖI KHỞI ĐỘNG SERVICE:\n", f.read())
    raise RuntimeError("❌ Không thể kết nối tới Service See-through.")

# 4. Tiến hành Convert
print(f"\n🚀 Bắt đầu quá trình Convert ảnh: {INPUT_FILE}...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_inp = f"/content/output_1788663167379/{INPUT_FILE.rsplit('.', 1)[0]}.inp"
out_live2d = f"/content/output_1788663167379/live2d"

env_run = env_srv.copy()
env_run['IMAGE2LIVE2D_DECOMPOSE_URL'] = 'http://localhost:8765'

cmd = [
    sys.executable, "-m", "image2live2d", "--image", INPUT_PATH,
    "--decompose-url", "http://localhost:8765",
    "-o", out_inp, "--live2d", out_live2d
]

result = subprocess.run(cmd, cwd="/content/image2live2d", env=env_run, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print("❌ LỖI CONVERT (CLI):")
    print(result.stderr[-3000:])

    # Đọc và hiển thị nội dung lỗi thực tế từ file log của server
    log_file.flush()
    with open("/content/seethrough_service.log", "r") as f:
        logs = f.read()
        print("\n=== CHI TIẾT LOG LỖI PHÍA SERVER ===\n")
        print(logs[-3000:])
else:
    print("✅ Convert thành công! Đang tiến hành đóng gói file zip...")

    # 5. Đóng gói & Tải xuống
    output_zip = f"/content/output_1788663167379.zip"
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, dirs, filenames in os.walk("/content/output_1788663167379"):
            for fname in filenames:
                fpath = os.path.join(root, fname)
                arcname = os.path.relpath(fpath, "/content/output_1788663167379")
                zf.write(fpath, arcname)

    size = os.path.getsize(output_zip) / 1024 / 1024
    print(f"📦 Đã đóng gói xong: {output_zip} ({size:.2f} MB)")
    files.download(output_zip)
    print("📥 Trình duyệt của bạn sẽ tự động tải file zip về máy!")

log_file.close()

🔄 Đã khôi phục trạng thái mã nguồn gốc sạch!
✅ Kiểm tra môi trường: Torch CUDA hoạt động bình thường trên Kernel chính!
🔄 Khởi động dịch vụ nền See-through...
.✅ Service đã khởi động thành công!

🚀 Bắt đầu quá trình Convert ảnh: 1788663167379 (3).png...
decomposing /content/1788663167379 (3).png via http://localhost:8765 (remote GPU)…

❌ LỖI CONVERT (CLI):
error: decompose service 500: Internal Server Error


=== CHI TIẾT LOG LỖI PHÍA SERVER ===

hon3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 164, in __call__
    await self.app(scope, receive, _send)
  File "/usr/loc